In [1]:
# ================================================================
# REAL-TIME LOG ANALYTICS PLATFORM USING PYSPARK
# COMPLETE END-TO-END SINGLE FILE PROJECT
# ================================================================

# ================================================================
# IMPORTS
# ================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime
import shutil

import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] += ";C:\\hadoop\\bin"

# ================================================================
# CREATE SPARK SESSION
# ================================================================

spark = (
    SparkSession.builder
    .appName("RealTimeLogAnalyticsPlatform")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("=" * 70)
print("REAL-TIME LOG ANALYTICS PLATFORM STARTED")
print("=" * 70)

# ================================================================
# READ RAW LOG FILE
# ================================================================

# Sample logs.txt format:
#
# 2026-05-17 10:15:21 INFO user123 LOGIN_SUCCESS 200 120
# 2026-05-17 10:15:25 ERROR user456 PAYMENT_FAILED 500 300
# 2026-05-17 10:15:30 WARN user789 API_TIMEOUT 408 2000

source_file = "./input/logs.txt"

if not os.path.exists(source_file):

    print("\nNO INPUT FILE FOUND")
    print("Pipeline stopped gracefully")

else:
    print("\nFILE AVAILABLE")
    
    
    raw_df = spark.read.text(source_file)
    
    print("\nRAW LOG DATA")
    raw_df.show(truncate=False)
    
    # ================================================================
    # PARSE LOG DATA
    # ================================================================
    
    logs_df = raw_df.select(
        split(col("value"), " ").getItem(0).alias("log_date"),
        split(col("value"), " ").getItem(1).alias("log_time"),
        split(col("value"), " ").getItem(2).alias("log_level"),
        split(col("value"), " ").getItem(3).alias("user_id"),
        split(col("value"), " ").getItem(4).alias("event_type"),
        split(col("value"), " ").getItem(5).alias("status_code"),
        split(col("value"), " ").getItem(6).alias("response_time_ms")
    )
    
    # ================================================================
    # CREATE TIMESTAMP COLUMN
    # ================================================================
    
    logs_df = logs_df.withColumn(
        "timestamp",
        to_timestamp(
            concat(
                col("log_date"),
                lit(" "),
                col("log_time")
            ),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    
    # ================================================================
    # TYPE CASTING
    # ================================================================
    
    logs_df = logs_df.withColumn(
        "status_code",
        col("status_code").cast("integer")
    )
    
    logs_df = logs_df.withColumn(
        "response_time_ms",
        col("response_time_ms").cast("integer")
    )
    
    print("\nPARSED LOG DATA")
    logs_df.show(truncate=False)
    
    print("\nSCHEMA")
    logs_df.printSchema()
    
    # ================================================================
    # REMOVE BAD RECORDS
    # ================================================================
    
    clean_df = logs_df.filter(
        col("response_time_ms") > 0
    )
    
    # ================================================================
    # REMOVE DUPLICATES
    # ================================================================
    
    clean_df = clean_df.dropDuplicates()
    
    # ================================================================
    # CACHE DATAFRAME
    # ================================================================
    
    clean_df.cache()
    clean_df.count()
    
    # ================================================================
    # TOTAL LOG COUNT
    # ================================================================
    
    total_logs = clean_df.count()
    
    print("\nTOTAL LOG COUNT:", total_logs)
    
    # ================================================================
    # ERROR ANALYTICS
    # ================================================================
    
    print("\nERROR ANALYTICS")
    
    error_df = (
        clean_df
        .filter(col("log_level") == "ERROR")
        .groupBy("event_type")
        .agg(
            count("*").alias("error_count")
        )
        .orderBy(desc("error_count"))
    )
    
    error_df.show()
    
    # ================================================================
    # WARNING ANALYTICS
    # ================================================================
    
    print("\nWARNING ANALYTICS")
    
    warning_df = (
        clean_df
        .filter(col("log_level") == "WARN")
        .groupBy("event_type")
        .agg(
            count("*").alias("warning_count")
        )
    )
    
    warning_df.show()
    
    # ================================================================
    # LOG LEVEL DISTRIBUTION
    # ================================================================
    
    print("\nLOG LEVEL DISTRIBUTION")
    
    log_level_df = (
        clean_df
        .groupBy("log_level")
        .agg(
            count("*").alias("total_logs")
        )
        .orderBy(desc("total_logs"))
    )
    
    log_level_df.show()
    
    # ================================================================
    # STATUS CODE ANALYTICS
    # ================================================================
    
    print("\nSTATUS CODE ANALYTICS")
    
    status_code_df = (
        clean_df
        .groupBy("status_code")
        .agg(
            count("*").alias("request_count")
        )
        .orderBy(desc("request_count"))
    )
    
    status_code_df.show()
    
    # ================================================================
    # API PERFORMANCE ANALYTICS
    # ================================================================
    
    print("\nAPI PERFORMANCE ANALYTICS")
    
    performance_df = (
        clean_df
        .groupBy("event_type")
        .agg(
            round(avg("response_time_ms"), 2).alias("avg_response_time"),
            min("response_time_ms").alias("min_response_time"),
            max("response_time_ms").alias("max_response_time"),
            count("*").alias("total_requests")
        )
        .orderBy(desc("avg_response_time"))
    )
    
    performance_df.show(truncate=False)
    
    # ================================================================
    # SLOW API DETECTION
    # ================================================================
    
    print("\nSLOW API DETECTION")
    
    slow_api_df = clean_df.filter(
        col("response_time_ms") > 1000
    )
    
    slow_api_df.show(truncate=False)
    
    # ================================================================
    # FAILED REQUEST ANALYTICS
    # ================================================================
    
    print("\nFAILED REQUEST ANALYTICS")
    
    failed_requests_df = clean_df.filter(
        col("status_code") >= 400
    )
    
    failed_requests_df.show(truncate=False)
    
    # ================================================================
    # USER ACTIVITY ANALYTICS
    # ================================================================
    
    print("\nUSER ACTIVITY ANALYTICS")
    
    user_activity_df = (
        clean_df
        .groupBy("user_id")
        .agg(
            count("*").alias("activity_count")
        )
        .orderBy(desc("activity_count"))
    )
    
    user_activity_df.show()
    
    # ================================================================
    # MOST ACTIVE USERS RANKING
    # ================================================================
    
    print("\nMOST ACTIVE USERS RANKING")
    
    window_spec = Window.orderBy(desc("activity_count"))
    
    ranked_users_df = user_activity_df.withColumn(
        "rank",
        dense_rank().over(window_spec)
    )
    
    ranked_users_df.show()
    
    # ================================================================
    # HOURLY TRAFFIC ANALYTICS
    # ================================================================
    
    print("\nHOURLY TRAFFIC ANALYTICS")
    
    hourly_traffic_df = (
        clean_df
        .withColumn("hour", hour(col("timestamp")))
        .groupBy("hour")
        .agg(
            count("*").alias("request_count")
        )
        .orderBy("hour")
    )
    
    hourly_traffic_df.show()
    
    # ================================================================
    # EVENT TYPE ANALYTICS
    # ================================================================
    
    print("\nEVENT TYPE ANALYTICS")
    
    event_df = (
        clean_df
        .groupBy("event_type")
        .agg(
            count("*").alias("event_count")
        )
        .orderBy(desc("event_count"))
    )
    
    event_df.show()
    
    # ================================================================
    # SESSIONIZATION
    # ================================================================
    
    print("\nSESSIONIZATION")
    
    session_window = Window.partitionBy("user_id").orderBy("timestamp")
    
    session_df = clean_df.withColumn(
        "previous_timestamp",
        lag("timestamp").over(session_window)
    )
    
    session_df = session_df.withColumn(
        "time_difference_seconds",
        unix_timestamp("timestamp") -
        unix_timestamp("previous_timestamp")
    )
    
    session_df.show(truncate=False)
    
    # ================================================================
    # REAL-TIME ALERT SIMULATION
    # ================================================================
    
    print("\nALERT MONITORING")
    
    error_count = (
        clean_df
        .filter(col("log_level") == "ERROR")
        .count()
    )
    
    warning_count = (
        clean_df
        .filter(col("log_level") == "WARN")
        .count()
    )
    
    slow_api_count = (
        clean_df
        .filter(col("response_time_ms") > 1000)
        .count()
    )
    
    if error_count > 2:
        print("ALERT: HIGH ERROR RATE DETECTED")
    
    if slow_api_count > 1:
        print("ALERT: MULTIPLE SLOW APIs DETECTED")
    
    if warning_count > 2:
        print("ALERT: HIGH WARNING COUNT")
    
    # ================================================================
    # SPARK SQL ANALYTICS
    # ================================================================
    
    print("\nSPARK SQL ANALYTICS")
    
    clean_df.createOrReplaceTempView("logs")
    
    spark.sql("""
    SELECT
        event_type,
        COUNT(*) AS total_events,
        ROUND(AVG(response_time_ms), 2) AS avg_response_time,
        MAX(response_time_ms) AS max_response_time
    FROM logs
    GROUP BY event_type
    ORDER BY avg_response_time DESC
    """).show(truncate=False)
    
    # ================================================================
    # ERROR PERCENTAGE
    # ================================================================
    
    print("\nERROR PERCENTAGE")
    
    error_percentage_df = (
        clean_df
        .groupBy("log_level")
        .agg(
            count("*").alias("log_count")
        )
        .withColumn(
            "percentage",
            round(
                (col("log_count") / total_logs) * 100,
                2
            )
        )
    )
    
    error_percentage_df.show()
    
    # ================================================================
    # TOP 5 SLOWEST EVENTS
    # ================================================================
    
    print("\nTOP 5 SLOWEST EVENTS")
    
    top_slowest_df = (
        clean_df
        .orderBy(desc("response_time_ms"))
        .limit(5)
    )
    
    top_slowest_df.show(truncate=False)
    
    # ================================================================
    # SAVE BRONZE LAYER
    # ================================================================
    
    print("\nSAVING BRONZE LAYER")
    
    raw_df.write.mode("overwrite").option("header", True).format("csv").save(
        "output/bronze/raw_logs"
    )
    
    # ================================================================
    # SAVE SILVER LAYER
    # ================================================================
    
    print("\nSAVING SILVER LAYER")
    
    clean_df.write.mode("overwrite").option("header", True).format("csv").save(
        "output/silver/clean_logs"
    )
    
    # ================================================================
    # SAVE GOLD LAYER
    # ================================================================
    
    print("\nSAVING GOLD LAYER")
    
    performance_df.write.mode("overwrite").option("header", True).format("csv").save(
        "output/gold/performance_analytics"
    )
    
    error_df.write.mode("overwrite").option("header", True).format("csv").save(
        "output/gold/error_analytics"
    )
    
    user_activity_df.write.mode("overwrite").option("header", True).format("csv").save(
        "output/gold/user_activity"
    )
    
    hourly_traffic_df.write.mode("overwrite").option("header", True).format("csv").save(
        "output/gold/hourly_traffic"
    )
    
    event_df.write.mode("overwrite").option("header", True).format("csv").save(
        "output/gold/event_analytics"
    )
    
    # ================================================================
    # AUDIT LOGGING
    # ================================================================
    
    print("\nAUDIT LOGGING")
    
    audit_df = (
        spark.range(1)
        .select(
            lit("RealTimeLogAnalyticsJob").alias("job_name"),
            lit(total_logs).alias("total_logs_processed"),
            lit(error_count).alias("total_errors"),
            lit(warning_count).alias("total_warnings"),
            lit(slow_api_count).alias("slow_api_count"),
            current_timestamp().alias("job_timestamp")
        )
    )
    
    audit_df.show(truncate=False)
    
    audit_df.write \
        .mode("overwrite") \
        .option("header", True) \
        .csv("output/audit/job_audit")
    
    # ================================================================
    # FINAL SUMMARY
    # ================================================================
    
    print("\n" + "=" * 70)
    print("FINAL SUMMARY")
    print("=" * 70)
    
    print(f"TOTAL LOGS: {total_logs}")
    print(f"ERROR LOGS: {error_count}")
    print(f"WARNING LOGS: {warning_count}")
    print(f"SLOW APIS: {slow_api_count}")
    
    print("=" * 70)
    print("REAL-TIME LOG ANALYTICS PIPELINE COMPLETED")
    print("=" * 70)
    
    # ================================================================
    # STOP SPARK SESSION
    # ================================================================
    
    spark.stop()
    
    
    # ================================================================
    # ARCHIVE THE LOG FILE
    # ================================================================
    
    archive_folder = "./input/Archival"
    
    # Timestamp for archive file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Archived filename
    archive_file = f"logs_{timestamp}.txt"
    
    # Full archive path
    archive_path = os.path.join(
        archive_folder,
        archive_file
    )
    
    # Move file to archive
    shutil.move(source_file, archive_path)
    
    print(f"File archived successfully")
    print(f"Archive Location: {archive_path}")

REAL-TIME LOG ANALYTICS PLATFORM STARTED

FILE AVAILABLE

RAW LOG DATA
+---------------------------------------------------------------------+
|value                                                                |
+---------------------------------------------------------------------+
|2026-05-17 10:15:21 INFO user123 LOGIN_SUCCESS 200 120               |
|2026-05-17 10:15:25 ERROR user456 PAYMENT_FAILED 500 300             |
|2026-05-17 10:15:30 WARN user789 API_TIMEOUT 408 2000                |
|2026-05-17 10:15:35 INFO user111 FILE_UPLOAD 201 450                 |
|2026-05-17 10:15:40 ERROR user222 DATABASE_CONNECTION_FAILED 500 1800|
|2026-05-17 10:15:45 INFO user333 PASSWORD_RESET 200 150              |
|2026-05-17 10:15:50 WARN user444 HIGH_MEMORY_USAGE 300 2500          |
|2026-05-17 10:15:55 INFO user555 PROFILE_UPDATED 200 90              |
|2026-05-17 10:16:00 ERROR user666 ORDER_PROCESSING_FAILED 502 3200   |
|2026-05-17 10:16:05 INFO user777 ITEM_ADDED_TO_CART 200 110     